# Getting Started with dataset-builder

This notebook walks you through building a complete text Q&A dataset about **solar energy** using `dataset-builder`.

## Prerequisites

- **Node.js** (v18+) and **npm** installed
- **Ollama** (optional, required only for LLM-based transformation)
- This notebook should be run from the `notebooks/` directory of the project

## What we'll build

1. **Scrape** web pages about solar energy using DuckDuckGo
2. **Transform** the scraped content into structured Q&A pairs
3. **Format** the dataset into Alpaca format with train/val/test splits

By the end, you'll have a ready-to-use ML training dataset.

In [ ]:
!npm install

## Step 1: Scraping

The `scrape` command searches the web and downloads page content. Key options:

- `--search` — the search query
- `--search-provider` — which search engine to use (`duckduckgo`, `google`, `bing`, etc.)
- `--search-count` — number of results to scrape
- `-y` — skip confirmation prompts

Output is saved to `output/task_<timestamp>/` with a `results.json` containing the scraped data.

In [ ]:
!npm start -- scrape --search "solar energy" --search-provider duckduckgo --search-count 3 -y

In [ ]:
import json
from pathlib import Path

# Find the latest task_* folder in the output directory
output_dir = Path("../output")
task_dirs = sorted(output_dir.glob("task_*"), key=lambda p: p.stat().st_mtime, reverse=True)

if not task_dirs:
    print("No task directories found. Did the scrape command complete successfully?")
else:
    latest_task = task_dirs[0]
    print(f"Latest task directory: {latest_task.name}")

    results_file = latest_task / "results.json"
    if results_file.exists():
        with open(results_file) as f:
            data = json.load(f)

        if isinstance(data, list):
            print(f"Total records: {len(data)}")
            if data:
                print(f"\nKeys in first record: {list(data[0].keys())}")
                print(f"\nFirst record (truncated):")
                preview = {k: (str(v)[:200] + "..." if len(str(v)) > 200 else v) for k, v in data[0].items()}
                print(json.dumps(preview, indent=2))
        else:
            print(f"Top-level keys: {list(data.keys())}")
    else:
        print(f"No results.json found in {latest_task}")

## Step 2: Transforming

The `transform` command converts raw scraped data into structured records using templates.

- `-i` — input directory (the scraped task folder)
- `-t text-qa` — use the text Q&A template, which generates question-answer pairs from text
- `--target` — the topic focus for generating relevant Q&A
- `-o` — output file path

The `text-qa` template uses an LLM to extract meaningful Q&A pairs from the scraped content.

In [ ]:
!npm start -- transform -i ../output/task_* -t text-qa --target "solar energy" -o ../output/nb_solar_qa.json

In [ ]:
import json
from pathlib import Path

qa_file = Path("../output/nb_solar_qa.json")

if qa_file.exists():
    with open(qa_file) as f:
        qa_data = json.load(f)

    print(f"Total Q&A records: {len(qa_data)}")
    if qa_data:
        print(f"\nKeys: {list(qa_data[0].keys())}")
        print(f"\nFirst record:")
        print(json.dumps(qa_data[0], indent=2))
else:
    print("nb_solar_qa.json not found. Did the transform command complete successfully?")

In [ ]:
!npm start -- format -i ../output/nb_solar_qa.json -f alpaca -o ../output/nb_solar_dataset/ --split 80:10:10

In [ ]:
import json
from pathlib import Path

dataset_dir = Path("../output/nb_solar_dataset")

if dataset_dir.exists():
    split_files = list(dataset_dir.glob("*.json"))
    print(f"Split files found: {len(split_files)}")

    for f in sorted(split_files):
        with open(f) as fh:
            records = json.load(fh)
        print(f"  {f.name}: {len(records)} records")

    # Show a sample Alpaca record from the first file
    if split_files:
        with open(sorted(split_files)[0]) as fh:
            sample = json.load(fh)
        if sample:
            print(f"\nSample Alpaca record from {sorted(split_files)[0].name}:")
            print(json.dumps(sample[0], indent=2))
else:
    print("nb_solar_dataset/ not found. Did the format command complete successfully?")

## Next Steps

You've built a complete dataset pipeline: scrape, transform, format. Here are more things to explore:

- **[02-data-generation.ipynb](02-data-generation.ipynb)** — Generate synthetic data with Faker and LLMs
- **Importing** — Bring in existing CSV, JSON, XML, or XLS files
- **Sanitizing** — Deduplicate, filter, and clean your datasets
- **Pipelines** — Chain multiple steps into automated workflows

Check the project README for the full command reference.

In [ ]:
import os
from pathlib import Path

# Cleanup: remove nb_ prefixed files and directories from output/
output_dir = Path("../output")
removed = []

for item in output_dir.iterdir():
    if item.name.startswith("nb_"):
        if item.is_file():
            item.unlink()
            removed.append(item.name)
        elif item.is_dir():
            import shutil
            shutil.rmtree(item)
            removed.append(item.name + "/")

if removed:
    print(f"Cleaned up {len(removed)} item(s):")
    for name in removed:
        print(f"  - {name}")
else:
    print("Nothing to clean up.")